# 04 - Processos de Negócio

## Descrição

Este notebook documenta os processos de negócio identificados no sistema. Cada processo é analisado em termos de propósito, gatilho, entradas, saídas, regras de validação, tratamento de exceções e dependências. Os processos cobrem desde a geração de dados sintéticos até a entrega de métricas analíticas no modelo dimensional.

---


## PN-01: Geração de Dados Sintéticos

### O que é?
Processo de criação de dados fictícios mas realistas de e-commerce (clientes, pedidos, produtos, etc.) para popular o MongoDB.

### Por que existe?
O projeto é acadêmico e não possui um sistema transacional real. Os dados sintéticos permitem demonstrar o pipeline completo com volume representativo (150.000 documentos no total).

### Como funciona?
1. **Geração**: Cada script `gerar_*.py` usa a biblioteca `Faker` (locale `pt_BR`) para criar 15.000 registros por coleção
2. **Determinismo**: Sementes fixas (`random.seed()`, `Faker.seed()`) garantem que a mesma execução produz os mesmos dados
3. **Coerência**: `updated_at` é sempre >= data de criação; chaves estrangeiras são sorteadas no intervalo `1..15000`
4. **Persistência**: CSVs são gravados em `dataset/arquivos_csv/` (não versionados, ~15MB)
5. **Carga**: `carregar_mongo.py` lê os CSVs, converte tipos BSON e insere no MongoDB em lotes de 5.000

### Entradas

| Entrada | Tipo | Origem |
|---------|------|--------|
| Parâmetros de geração | Código | Scripts `gerar_*.py` |
| Sementes aleatórias | Configuração | Hardcoded nos scripts |
| CSVs existentes | Arquivo | `dataset/arquivos_csv/` (se já gerados) |

### Saídas

| Saída | Tipo | Destino |
|-------|------|---------|
| Arquivos CSV | `*.csv` | `dataset/arquivos_csv/` |
| Coleções MongoDB | BSON | `mongodb` container (localhost:27017) |

### Regras de Validação

- Cada coleção recebe `$jsonSchema` na criação (pode ser desativado via `--no-validator`)
- Índices criados: PK única, FKs, `updated_at` ascendente
- Dados vazios convertidos para `null` (não string vazia)

### Exceções

| Exceção | Tratamento |
|---------|-----------|
| MongoDB indisponível | Script falha com exceção de conexão |
| CSV faltando | `gerar_dados.py` é chamado automaticamente |
| Coleção já existe | `drop_collection` + `create_collection` (idempotente) |

### Dependências

- Docker Compose com MongoDB rodando
- Python 3.11+ com dependências do grupo `dataset`
- Variáveis `MONGO_URI` e `MONGO_DB` configuradas

### Diagrama de Sequência

```mermaid
sequenceDiagram
    participant Dev as Desenvolvedor
    participant Gerar as Gerador de Dados
    participant CSV as CSVs
    participant Carregar as Carregar MongoDB
    participant Mongo as MongoDB

    Dev->>Gerar: Executa gerar_dados.py
    Gerar->>CSV: Verifica se CSVs existem
    alt CSVs ausentes
        Gerar->>Gerar: Gera 15K registros por coleção (Faker)
        Gerar->>CSV: Salva arquivos CSV
    end
    Dev->>Carregar: Executa carregar_mongo.py
    Carregar->>CSV: Lê CSVs de todas as coleções
    loop Para cada coleção
        Carregar->>Carregar: Converte tipos (int, float, bool, date)
        Carregar->>Carregar: Cria $jsonSchema e índices
        Carregar->>Mongo: drop_collection + create_collection
        Carregar->>Mongo: insert_many (batch 5000)
    end
    Mongo-->>Carregar: Confirma inserção
    Carregar-->>Dev: Relatório de carga
```

---

## PN-02: Extração Incremental MongoDB → Landing

### O que é?
Processo de extração de documentos do MongoDB que foram criados ou alterados desde a última execução, gravando-os em JSON no Data Lake.

### Por que existe?
Minimizar a carga no MongoDB e o volume de transferência, processando apenas o delta de mudanças. Permite near real-time analytics.

### Como funciona?
1. **Validação**: Verifica se o bucket Landing existe no MinIO/S3
2. **Validação de origem**: Verifica se as 10 coleções existem no MongoDB
3. **Recuperação de checkpoint**: Lê Airflow Variable com o maior `updated_at` processado por coleção
4. **Filtro incremental**: `updated_at >= checkpoint - overlap_hours` (overlap de 24h para evitar perder registros em borda)
5. **Extração em lote**: Cursor MongoDB com `batch_size=1000`, ordenado por `updated_at`
6. **Serialização**: Cada documento é serializado para MongoDB Extended JSON Canonical
7. **Gravação**: Arquivo JSON Lines gravado no MinIO em prefixo particionado por data e run_id
8. **Atualização de checkpoint**: Nova Variable com o maior `updated_at` extraído
9. **Manifesto**: JSON com totais, coleções e metadados da execução

### Entradas

| Entrada | Tipo | Origem |
|---------|------|--------|
| Documentos MongoDB | BSON | MongoDB Atlas / Local |
| Checkpoint anterior | Variável Airflow | `mongodb_landing_checkpoint__ecommerce_<colecao>` |
| Configuração de overlap | Variável de ambiente | `MONGO_CHECKPOINT_OVERLAP_HOURS=24` |

### Saídas

| Saída | Tipo | Destino |
|-------|------|---------|
| Arquivo JSON Lines | `*.json` | `s3://datalake/landing/ecommerce/<colecao>/extraction_date=.../run_id=.../part-00000.json` |
| Checkpoint atualizado | Variável Airflow | `mongodb_landing_checkpoint__ecommerce_<colecao>` |
| Manifesto de execução | JSON | `s3://datalake/landing/_control/mongodb_to_landing/.../manifest.json` |

### Regras de Validação

- Bucket Landing deve existir (falha se ausente)
- Coleções devem existir no MongoDB (falha se ausente)
- Overlap hours não pode ser negativo
- Documentos vazios (zero registros) não geram arquivo (mas checkpoint não avança se nada mudou)

### Exceções

| Exceção | Tratamento |
|---------|-----------|
| Bucket Landing inexistente | `AirflowException` — falha imediata |
| Coleção MongoDB inexistente | `AirflowException` — falha imediata |
| Erro de conexão MongoDB | Retry com backoff (2 retries, 2 min delay) |
| Erro de gravação S3 | Retry com backoff (2 retries, 2 min delay) |

### Dependências

- MongoDB acessível (local ou Atlas)
- MinIO/S3 acessível
- Estrutura Landing criada (`scripts/criar_estrutura_landing.py`)
- Airflow Connection `mongodb_atlas` configurada
- Airflow Connection `minio_s3` configurada

### Diagrama de Sequência

```mermaid
sequenceDiagram
    participant AF as Airflow Scheduler
    participant DAG as DAG mongodb_to_landing
    participant Mongo as MongoDB
    participant Var as Airflow Variables
    participant S3 as MinIO/S3

    AF->>DAG: Dispara execução (a cada 15 min)
    DAG->>S3: Valida bucket Landing existe
    DAG->>Mongo: Valida coleções existem
    loop Para cada coleção (expand)
        DAG->>Var: Lê checkpoint anterior
        DAG->>Mongo: find(updated_at >= checkpoint - 24h)
        Mongo-->>DAG: Retorna cursor (batch 1000)
        DAG->>DAG: Serializa para JSON Lines
        alt Documentos encontrados
            DAG->>S3: PUT part-00000.json
            DAG->>Var: Atualiza checkpoint
        end
    end
    DAG->>S3: PUT manifest.json
    DAG-->>AF: Status de conclusão
```

---

## PN-03: Conversão Landing → Bronze

### O que é?
Processo de conversão dos arquivos JSON da Landing para tabelas Delta Lake na camada Bronze, adicionando metadados de auditoria.

### Por que existe?
Transformar dados semi-estruturados (JSON com tipos BSON) em um formato analítico eficiente (Delta Lake), com rastreabilidade completa de linhagem.

### Como funciona?
1. **Validação**: Verifica bucket acessível e marcadores `_READY` da Bronze
2. **Validação de entrada**: Lista arquivos JSON na Landing para cada coleção (falha se zero)
3. **Spark Submit**: Submete job PySpark com pacotes Delta Lake e Hadoop-AWS
4. **Leitura**: Job lê JSON Lines com `recursiveFileLookup=true` e `pathGlobFilter=*.json`
5. **Metadados**: Extrai `extraction_date` e `run_id` do caminho do arquivo via regex
6. **Auditoria**: Adiciona `_bronze_source_file`, `_bronze_extraction_date`, `_bronze_landing_run_id`, `_bronze_airflow_run_id`, `_bronze_ingested_at`, `ingestion_date`
7. **Idempotência**: Se tabela Bronze já existe, faz left anti-join em `_bronze_source_file` para evitar reprocessar arquivos já ingeridos
8. **Gravação**: Append em Delta Lake com `mergeSchema=true`, particionado por `ingestion_date`
9. **Manifesto**: JSON com totais de linhas escritas, arquivos processados e status

### Entradas

| Entrada | Tipo | Origem |
|---------|------|--------|
| Arquivos JSON | `*.json` | `s3://datalake/landing/ecommerce/<colecao>/` |
| Marcadores Bronze | Objeto S3 | `s3://datalake/bronze/ecommerce/<colecao>/_READY` |
| Configuração Spark | Variável de ambiente | `SPARK_PACKAGES`, `SPARK_S3_ENDPOINT` |

### Saídas

| Saída | Tipo | Destino |
|-------|------|---------|
| Tabela Delta Bronze | Delta Lake | `s3://datalake/bronze/ecommerce/<colecao>/` |
| Manifesto de execução | JSON | `s3://datalake/bronze/_control/landing_to_bronze/.../manifest.json` |

### Regras de Validação

- Bucket deve ser acessível
- Marcadores `_READY` da Bronze devem existir para todas as coleções
- Deve haver pelo menos um arquivo JSON na Landing para cada coleção
- Arquivos já processados (com `_bronze_source_file` igual) são ignorados

### Exceções

| Exceção | Tratamento |
|---------|-----------|
| Bucket inacessível | `AirflowException` |
| Marcador Bronze ausente | `AirflowException` |
| Sem arquivos JSON na Landing | `AirflowException` |
| Falha no Spark job | Retry (2x, 5 min) + timeout de 2h |

### Dependências

- Execução bem-sucedida de `mongodb_to_landing` (arquivos JSON disponíveis)
- Estrutura Bronze criada (`scripts/criar_estrutura_bronze.py`)
- Spark configurado com Delta Lake e S3A

### Diagrama de Sequência

```mermaid
sequenceDiagram
    participant AF as Airflow Scheduler
    participant DAG as DAG landing_to_bronze
    participant S3 as MinIO/S3
    participant Spark as Spark Job

    AF->>DAG: Dispara execução (offset 5 min)
    DAG->>S3: Valida bucket e marcadores _READY
    DAG->>S3: Lista arquivos JSON na Landing
    DAG->>Spark: spark-submit landing_to_bronze.py
    Spark->>S3: READ JSON Lines (S3A)
    Spark->>Spark: Extrai metadata do path (regex)
    Spark->>Spark: Adiciona colunas de auditoria
    alt Tabela Bronze já existe
        Spark->>Spark: Left anti-join em _bronze_source_file
    end
    Spark->>S3: WRITE Delta Lake (append, partitionBy ingestion_date)
    Spark->>S3: PUT manifest.json
    Spark-->>DAG: Retorna status
    DAG-->>AF: Status de conclusão
```

---

## PN-04: Limpeza e Conformação Bronze → Silver

### O que é?
Processo de transformação dos dados brutos da Bronze em dados limpos, tipados, deduplicados e validados na Silver. É o coração da qualidade de dados do pipeline.

### Por que existe?
Dados de origem NoSQL são inconsistentes por natureza: tipos heterogêneos, duplicatas, valores fora de domínio, chaves estrangeiras órfãs. A Silver garante que apenas dados confiáveis cheguem à camada analítica.

### Como funciona?

#### 4.1. Leitura e Conversão de Tipos
1. Lê tabela Delta Bronze via `spark.read.format("delta")`
2. Converte tipos BSON para tipos Spark nativos:
   - `NumberInt` → `int` / `long`
   - `ISODate` → `timestamp` / `date`
   - `NumberDouble` → `double` / `decimal`
3. Aplica normalização de strings (`trim`, `lower`, `upper`, `digits-only`)

#### 4.2. Deduplicação
1. Agrupa por chave primária
2. Mantém o registro mais recente por critério de desempate:
   - `updated_at` DESC (mais recente primeiro)
   - `_bronze_ingested_at` DESC (mais recentemente ingerido)
   - `_bronze_source_file` DESC (determinístico final)
3. Registros duplicados removidos são contabilizados no manifesto

#### 4.3. Validação de Qualidade
Regras definidas em `ENTITY_RULES` (`dags/lib/bronze_silver.py`):

| Regra | Exemplo |
|-------|---------|
| **Obrigatoriedade** | `nome` não pode ser nulo ou vazio |
| **Domínio fechado** | `genero` ∈ {M, F, Outro} |
| **Padrão regex** | `cpf` deve ter 11 dígitos; `email` deve ter formato válido |
| **Range numérico** | `preco` >= 0; `desconto_percentual` <= 100 |
| **Unicidade de negócio** | CPF único entre clientes; um pagamento aprovado por pedido |

#### 4.4. Integridade Referencial
1. Para cada chave estrangeira definida na regra da entidade:
   - Faz broadcast join com a tabela Silver pai já processada
   - Se `required=True`: registros órfãos são removidos
   - Se `required=False`: registros com FK nula são preservados; FK não-nula inválida é removida
2. Registros removidos por violação de FK são contabilizados

#### 4.5. MERGE Incremental
1. Compara registros válidos com tabela Silver existente (se houver)
2. **Insere** registros novos (PK inexistente)
3. **Atualiza** registros alterados (`updated_at` maior OU hash diferente)
4. **Ignora** registros inalterados
5. Métricas: `inserted`, `updated`, `unchanged`, `rows_written`

#### 4.6. Quality Log
Registros rejeitados por regras de unicidade de negócio são gravados em `silver/_control/quality_log/` com metadados completos.

### Entradas

| Entrada | Tipo | Origem |
|---------|------|--------|
| Tabela Delta Bronze | Delta Lake | `s3://datalake/bronze/ecommerce/<tabela>/` |
| Regras de entidade | Código Python | `dags/lib/bronze_silver.py` (ENTITY_RULES) |
| Tabelas Silver processadas | Delta Lake | Usadas para validação de FK |

### Saídas

| Saída | Tipo | Destino |
|-------|------|---------|
| Tabela Delta Silver | Delta Lake | `s3://datalake/silver/ecommerce/<tabela>/` |
| Quality Log | Delta Lake | `s3://datalake/silver/_control/quality_log/` |
| Manifesto de execução | JSON | `s3://datalake/silver/_control/bronze_to_silver/.../manifest.json` |

### Regras de Validação

- Tabela Bronze deve existir (`_delta_log` presente)
- Marcadores Silver `_READY` devem existir
- Dependências Silver (tabelas pai de FK) devem estar processadas na ordem correta

### Exceções

| Exceção | Tratamento |
|---------|-----------|
| Tabela Bronze ausente | `AirflowException` |
| Marcador Silver ausente | `AirflowException` |
| Dependência Silver não processada | `parse_pipeline_tables` rejeita ordem inválida |
| Falha no Spark job | Retry (2x, 5 min) + timeout 2h |

### Dependências

- Execução bem-sucedida de `landing_to_bronze` (tabelas Bronze populadas)
- Estrutura Silver criada (`scripts/criar_estrutura_silver.py`)
- Ordem de processamento respeita dependências de FK (ex: `clientes` antes de `pedidos`)

### Diagrama de Sequência

```mermaid
sequenceDiagram
    participant AF as Airflow Scheduler
    participant DAG as DAG bronze_to_silver
    participant S3 as MinIO/S3
    participant Spark as Spark Job

    AF->>DAG: Dispara execução (offset 10 min)
    DAG->>S3: Valida bucket, marcadores _READY, _delta_log Bronze
    DAG->>Spark: spark-submit bronze_to_silver.py
    Spark->>S3: READ Delta Bronze (S3A)
    Spark->>Spark: Converte tipos BSON → Spark
    Spark->>Spark: Normaliza strings (trim, lower, digits)
    Spark->>Spark: Deduplica por PK (updated_at DESC)
    Spark->>Spark: Valida qualidade (required, enum, regex, range)
    Spark->>Spark: Valida unicidade de negócio (CPF, pagamento aprovado)
    Spark->>S3: WRITE rejeições para quality_log
    Spark->>Spark: Valida integridade referencial (broadcast join)
    Spark->>S3: MERGE Delta Silver (insert/update/ignore)
    Spark->>S3: PUT manifest.json
    Spark-->>DAG: Retorna status
    DAG-->>AF: Status de conclusão
```

---

## PN-05: Modelagem Dimensional Silver → Gold

### O que é?
Processo de transformação dos dados limpos da Silver em um modelo dimensional (Kimball) com dimensões SCD Tipo 2 e fatos analíticos.

### Por que existe?
Entregar dados estruturados para análise de negócio com histórico versionado e métricas pré-calculadas, permitindo consultas OLAP eficientes.

### Como funciona?

#### 5.1. Construção de Dimensões

| Dimensão | Tipo SCD | Fonte Silver | Atributos |
|----------|----------|--------------|-----------|
| `dim_tempo` | Static (tipo 0) | Datas de todas as tabelas | data_key, ano, semestre, trimestre, mes, mes_nome, semana_ano, dia_mes, dia_semana, fim_de_semana |
| `dim_cliente` | Tipo 2 | `clientes` | cliente_key, nome, genero, data_nascimento, cidade, estado, data_cadastro |
| `dim_produto` | Tipo 2 | `produtos` + `categorias` + `fornecedores` | produto_key, nome, descricao, marca, preco, estoque, categoria_key, nome_categoria, fornecedor_key, nome_fornecedor |
| `dim_cupom` | Tipo 2 | `cupons` | cupom_key, codigo, desconto_percentual, valor_minimo, data_validade, ativo |

#### 5.2. Implementação SCD Tipo 2
1. Calcula `dw_record_hash` (SHA256 dos atributos de negócio) para cada registro
2. Compara com versão atual (`dw_is_current=true`) na dimensão Gold
3. Se hash mudou: expira versão anterior (`dw_valid_to = now`, `dw_is_current = false`) e insere nova versão (`dw_valid_from = now`, `dw_is_current = true`)
4. Se hash igual: não faz nada (idempotente)
5. Surrogate key (`cliente_sk`) = SHA256(natural_key + dw_valid_from)

#### 5.3. Construção de Fatos

| Fato | Fontes Silver | Métricas | Partição |
|------|---------------|----------|----------|
| `fato_vendas` | `itens_pedido` + `pedidos` | quantidade, valor_unitario, desconto, valor_bruto, valor_desconto, receita_liquida | `ano` |
| `fato_pagamentos` | `pagamentos` + `pedidos` | valor, valor_aprovado, parcelas, quantidade_pagamentos | `ano` |
| `fato_entregas` | `entregas` + `pedidos` | prazo_previsto_dias, prazo_real_dias, atraso_dias, entrega_no_prazo, quantidade_entregas | `ano` |
| `fato_avaliacoes` | `avaliacoes` | nota, comentario, avaliacao_positiva, quantidade_avaliacoes | `ano` |

#### 5.4. Point-in-Time Join (Surrogate Keys nos Fatos)
1. Para cada fato, identifica a data do evento (ex: `data_pedido` em `fato_vendas`)
2. Faz join com a dimensão SCD2 filtrando: `event_date >= dw_valid_from AND event_date < dw_valid_to`
3. Anexa a surrogate key vigente naquela data (`cliente_sk`, `produto_sk`, `cupom_sk`)
4. Isso garante que o fato aponte para a versão correta do cliente/produto/cupom na época da compra

#### 5.5. MERGE e Sincronização
- **Dimensões SCD2**: Merge com `whenMatchedUpdate` (expira versão antiga) + `whenNotMatchedInsert` (nova versão)
- **Fatos**: Merge com `whenMatchedUpdateAll` (atualiza se hash mudou) + `whenNotMatchedInsertAll` (insere novo) + `whenNotMatchedBySourceDelete` (remove se não existe mais na Silver)

### Entradas

| Entrada | Tipo | Origem |
|---------|------|--------|
| Tabelas Delta Silver | Delta Lake | `s3://datalake/silver/ecommerce/<tabela>/` |
| Modelos Gold | Código Python | `dags/lib/silver_gold.py` (GOLD_MODELS) |
| Links fato-dimensão | Código Python | `spark_jobs/silver_to_gold.py` (FACT_DIMENSION_LINKS) |

### Saídas

| Saída | Tipo | Destino |
|-------|------|---------|
| Dimensões Delta Gold | Delta Lake | `s3://datalake/gold/ecommerce/dim_*/` |
| Fatos Delta Gold | Delta Lake | `s3://datalake/gold/ecommerce/fato_*/` |
| Manifesto de execução | JSON | `s3://datalake/gold/_control/silver_to_gold/.../manifest.json` |

### Regras de Validação

- Tabelas Silver devem existir (`_delta_log` presente) para todas as fontes necessárias
- Marcadores Gold `_READY` devem existir
- Dimensões são processadas antes dos fatos (para permitir point-in-time join)

### Exceções

| Exceção | Tratamento |
|---------|-----------|
| Tabela Silver ausente | `AirflowException` |
| Marcador Gold ausente | `AirflowException` |
| Fato gera zero registros | `ValueError` — falha imediata |
| Falha no Spark job | Retry (2x, 5 min) + timeout 2h |

### Dependências

- Execução bem-sucedida de `bronze_to_silver` (tabelas Silver populadas)
- Estrutura Gold criada (`scripts/criar_estrutura_gold.py`)
- Ordem: dimensões primeiro, fatos depois (point-in-time join requer dimensões prontas)

### Diagrama de Sequência

```mermaid
sequenceDiagram
    participant AF as Airflow Scheduler
    participant DAG as DAG silver_to_gold
    participant S3 as MinIO/S3
    participant Spark as Spark Job

    AF->>DAG: Dispara execução (offset 15 min)
    DAG->>S3: Valida bucket, marcadores _READY, _delta_log Silver
    DAG->>Spark: spark-submit silver_to_gold.py
    Spark->>S3: READ Delta Silver (S3A)
    Note over Spark: Construção de Dimensões
    Spark->>Spark: Build dim_tempo (explode de sequência de datas)
    Spark->>Spark: Build dim_cliente, dim_produto, dim_cupom (SCD2)
    Spark->>Spark: Calcula dw_record_hash por dimensão
    Spark->>S3: MERGE dimensões SCD2 (expira + insere)
    Note over Spark: Construção de Fatos
    Spark->>Spark: Build fato_vendas (join itens_pedido + pedidos)
    Spark->>Spark: Build fato_pagamentos (join pagamentos + pedidos)
    Spark->>Spark: Build fato_entregas (join entregas + pedidos)
    Spark->>Spark: Build fato_avaliacoes (join avaliacoes)
    Spark->>Spark: Point-in-time join: anexa SKs vigentes
    Spark->>Spark: Calcula métricas (receita, atraso, aprovação, etc.)
    Spark->>S3: MERGE fatos (insert/update/delete)
    Spark->>S3: PUT manifest.json
    Spark-->>DAG: Retorna status
    DAG-->>AF: Status de conclusão
```

---

## PN-06: Consumo Analítico (Dashboard/BI)

### O que é?
Processo de consulta aos dados da camada Gold por ferramentas de BI para geração de dashboards e relatórios.

### Por que existe?
Entregar valor de negócio aos stakeholders através de métricas e KPIs calculados a partir dos dados processados.

### Como funciona?
1. Ferramenta de BI (ou Jupyter, ou SQL engine) conecta-se ao Data Lake via S3A ou Delta Lake connector
2. Consulta dimensões e fatos usando SQL
3. O particionamento por `ano` nos fatos otimiza consultas temporais
4. SCD Tipo 2 permite análise temporal: "como era o cliente no momento da compra" vs "como é agora"

### KPIs e Métricas Disponíveis

| Categoria | Métrica | Fonte |
|-----------|---------|-------|
| **Vendas** | Receita bruta, líquida, desconto | `fato_vendas` |
| **Vendas** | Quantidade de itens vendidos | `fato_vendas.quantidade` |
| **Pagamentos** | Valor aprovado | `fato_pagamentos.valor_aprovado` |
| **Pagamentos** | Taxa de aprovação | `SUM(valor_aprovado) / SUM(valor)` |
| **Logística** | Prazo real vs. previsto | `fato_entregas.prazo_real_dias`, `prazo_previsto_dias` |
| **Logística** | Percentual de entregas no prazo | `AVG(entrega_no_prazo)` |
| **Satisfação** | Nota média | `fato_avaliacoes.nota` |
| **Satisfação** | Percentual de avaliações positivas | `AVG(avaliacao_positiva)` |

### Entradas

| Entrada | Tipo | Origem |
|---------|------|--------|
| Dimensões e Fatos Gold | Delta Lake | `s3://datalake/gold/ecommerce/` |
| Ferramenta de BI | Aplicação | Power BI, Tableau, Jupyter, etc. |

### Saídas

| Saída | Tipo | Destino |
|-------|------|---------|
| Dashboards | Visual | Usuários de negócio |
| Relatórios | Documento | Stakeholders |